In [ ]:
# Cài đặt thư viện Unsloth trên Google Colab
!pip install -q unsloth

import json
import os
import torch
from google.colab import drive
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

# 1. Kết nối Google Drive
drive.mount('/content/drive')

TRAIN_DATA_PATH = "/content/drive/MyDrive/NovelAI/dataset/data_train.jsonl"
LORA_SAVE_DIR = "/content/drive/MyDrive/NovelAI/lora_adapter"

# 2. Tải mô hình gốc Qwen3-4B-Instruct ở chế độ nén 4-bit
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. Gắn các nhánh phụ QLoRA vào 7 lớp mạng nơ-ron
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Đọc file JSON từ Google Drive và chuyển sang định dạng ChatML
raw_dataset = load_dataset("json", data_files=TRAIN_DATA_PATH, split="train")

SYSTEM_PROMPT = (
    "Bạn là dịch giả tiểu thuyết chuyên nghiệp. Hãy dịch văn bản tiếng Anh ở phần 'input' sang tiếng Việt mượt mà, "
    "giữ nguyên dấu '...' cho độc thoại nội tâm, dấu 「...」 cho thông báo và tuân thủ tuyệt đối quy tắc xưng hô trong 'context'. "
    "Chỉ xuất ra bản dịch tiếng Việt, không giải thích gì thêm."
)

def format_json_to_chat(example):
    ctx_str = json.dumps(example["context"], ensure_ascii=False, indent=2)
    user_content = (
        f"### Nhiệm vụ:\n{example['instruction']}\n\n"
        f"### Ngữ cảnh & Quy tắc xưng hô (context):\n{ctx_str}\n\n"
        f"### Văn bản tiếng Anh cần dịch (input):\n{example['input']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False
    )
    return {"text": text}

train_dataset = raw_dataset.map(format_json_to_chat)

# 5. Cấu hình huấn luyện SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Chỉ tính hàm mất mát (Loss) trên phần trả lời tiếng Việt (output) của Assistant
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# 6. Bắt đầu huấn luyện
trainer_stats = trainer.train()

# 7. Lưu bộ trọng số LoRA Adapter (~100MB) vào Google Drive để Giai đoạn 3 sử dụng
os.makedirs(LORA_SAVE_DIR, exist_ok=True)
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)
print(f"Đã lưu thành công LoRA Adapter tại: {LORA_SAVE_DIR}")

In [ ]:
# Chuyển mô hình sang chế độ dịch nhanh
FastLanguageModel.for_inference(model)

test_sample = {
    "instruction": "Dịch truyện tiếng Anh sang tiếng Việt",
    "context": {
        "speaker": "Yuuka",
        "listener": "Sensei",
        "scene": "hostile",
        "pronouns": {"you": "ngươi"}
    },
    "input": "\"Why are you alive? Could it be, you weren't there?\""
}

user_test = (
    f"### Nhiệm vụ:\n{test_sample['instruction']}\n\n"
    f"### Ngữ cảnh & Quy tắc xưng hô (context):\n{json.dumps(test_sample['context'], ensure_ascii=False, indent=2)}\n\n"
    f"### Văn bản tiếng Anh cần dịch (input):\n{test_sample['input']}"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_test}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.2, use_cache=True)

result_vi = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("Bản dịch của AI:", result_vi)